# Notebook 14: Investment Agent Linker

## Three Agents, One Notebook

This notebook brings together the three investment agents from Notebooks 11–13
into a single unified workspace. The portfolio you build conversationally in
**Agent 1** flows automatically into **Agents 2 and 3**.

## Architecture
```
┌──────────────────────────────────────────────────────────────────┐
│                   INVESTMENT AGENT LINKER                        │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  Agent 1: Portfolio Designer (NB11)                              │
│    Conversational profiling → portfolio_generation tool          │
│              │                                                   │
│              ▼                                                   │
│     ┌─── portfolio.json ───┐                                     │
│     │   (shared contract)  │                                     │
│     └──┬───────────────┬───┘                                     │
│        │               │                                         │
│        ▼               ▼                                         │
│  Agent 2: Analytics  Agent 3: Education                          │
│  (NB12)              (NB13)                                      │
│  Full scoring +      Multi-turn chat                             │
│  risk analysis       with portfolio                              │
│  (one-shot report)   context                                     │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

## How to Use This Notebook
1. Run the **Setup** cells (install, imports, config)
2. **Agent 1**: Chat with the profiling agent to design your portfolio
3. **Agent 2**: Run the full analytics pipeline on your portfolio
4. **Agent 3**: Chat about your portfolio and investment concepts
5. Come back to **Agent 1** to redesign, then re-run Agents 2 & 3

| Individual Notebooks (11–13) | This Linker (14) |
|------------------------------|---------------|
| Separate files | All in one place |
| JSON file hand-off | Shared in-memory + JSON |
| Run notebooks sequentially | Jump between agents freely |
| Duplicate setup code | Single shared setup |


---
# Setup (Run Once)
---

## Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph python-dotenv pandas pydantic yfinance google-search-results --quiet

print("Packages installed")

## Imports

In [ ]:
import json
import os
from typing import Annotated, Literal, Sequence

import pandas as pd
import numpy as np
import yfinance as yf
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from IPython.display import Image, display, Markdown

from ai_course_utils import load_llm_from_env, display_config

# Load environment
load_dotenv()
load_dotenv('../.env')

print("Imports successful")

## Configuration

In [ ]:
display_config()

## Shared Resources

These constants and helpers are used across all three agents.
Run these cells once — they provide the foundation for everything below.

In [ ]:
# ============================================================================
# SHARED: Portfolio file path and in-memory state
# ============================================================================
PORTFOLIO_FILE = "../data/outputs/portfolio.json"

# This variable is shared across all three agents
SHARED_PORTFOLIO = None


def save_portfolio(portfolio: dict):
    """Save portfolio to JSON and update the shared variable."""
    global SHARED_PORTFOLIO
    SHARED_PORTFOLIO = portfolio
    os.makedirs(os.path.dirname(PORTFOLIO_FILE), exist_ok=True)
    with open(PORTFOLIO_FILE, "w") as f:
        json.dump(portfolio, f, indent=2)
    print(f"Portfolio saved to {PORTFOLIO_FILE}")


def load_portfolio_from_file() -> dict | None:
    """Load portfolio from JSON file if it exists."""
    global SHARED_PORTFOLIO
    if os.path.exists(PORTFOLIO_FILE):
        with open(PORTFOLIO_FILE) as f:
            SHARED_PORTFOLIO = json.load(f)
        return SHARED_PORTFOLIO
    return None


def show_portfolio(portfolio: dict | None = None):
    """Display the current portfolio."""
    p = portfolio or SHARED_PORTFOLIO
    if not p:
        print("No portfolio loaded. Use Agent 1 to create one.")
        return
    print(f"Portfolio: {p['name']}")
    print(f"Holdings:  {len(p['holdings'])}")
    total = sum(h['allocation_pct'] for h in p['holdings'])
    print(f"Total:     {total}%")
    print()
    for h in p['holdings']:
        print(f"  {h['ticker']:<6} {h['allocation_pct']:>5.1f}%  {h['company_name']}")


print("Shared portfolio functions defined")

In [ ]:
# ============================================================================
# SHARED: S&P 500 Sector Benchmark
# ============================================================================
SP500_SECTOR_WEIGHTS = {
    "Technology": 31.0,
    "Healthcare": 12.0,
    "Financial Services": 13.0,
    "Consumer Cyclical": 10.0,
    "Communication Services": 9.0,
    "Industrials": 8.0,
    "Consumer Defensive": 6.0,
    "Energy": 4.0,
    "Utilities": 3.0,
    "Real Estate": 2.0,
    "Basic Materials": 2.0,
}

print("S&P 500 sector benchmark loaded")

In [ ]:
# ============================================================================
# SHARED: Market data helpers (used by Agents 1 and 2)
# ============================================================================

def _scalar(x):
    """Convert numpy/pandas scalar to plain Python float."""
    try:
        if hasattr(x, "item"):
            return x.item()
        if pd.isna(x):
            return None
        return float(x)
    except (TypeError, ValueError):
        return None


def fetch_price_history(ticker: str, period: str = "1y") -> pd.DataFrame:
    """Fetch OHLCV history from yfinance."""
    try:
        df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df
    except Exception as e:
        print(f"    Warning: yfinance download failed for {ticker}: {e}")
        return pd.DataFrame()


def fetch_fundamentals(ticker: str) -> dict:
    """Fetch fundamental metrics from yfinance."""
    try:
        info = yf.Ticker(ticker).info
        return {
            "company_name": info.get("shortName") or info.get("longName", ticker),
            "sector": info.get("sector", "Unknown"),
            "current_price": info.get("currentPrice") or info.get("regularMarketPrice"),
            "pe_ratio": info.get("trailingPE") or info.get("forwardPE"),
            "pb_ratio": info.get("priceToBook"),
            "roe": info.get("returnOnEquity"),
            "dividend_yield": info.get("dividendYield"),
            "beta": info.get("beta"),
            "market_cap": info.get("marketCap"),
            "52_week_high": info.get("fiftyTwoWeekHigh"),
            "52_week_low": info.get("fiftyTwoWeekLow"),
            "profit_margins": info.get("profitMargins"),
            "revenue_growth": info.get("revenueGrowth"),
            "debt_to_equity": info.get("debtToEquity"),
        }
    except Exception as e:
        print(f"    Warning: yfinance info failed for {ticker}: {e}")
        return {"company_name": ticker, "sector": "Unknown"}


def compute_technical_indicators(df: pd.DataFrame) -> dict | None:
    """Compute RSI, MACD, SMA from price history."""
    if df.empty or len(df) < 50:
        return None

    close = df["Close"]

    sma_50 = close.rolling(50).mean()
    sma_200 = close.rolling(200).mean() if len(df) >= 200 else pd.Series(dtype=float)

    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema_12 - ema_26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()

    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    latest = df.iloc[-1]

    return {
        "current_price": _scalar(latest["Close"]),
        "sma_50": _scalar(sma_50.iloc[-1]),
        "sma_200": _scalar(sma_200.iloc[-1]) if not sma_200.empty else None,
        "macd": _scalar(macd_line.iloc[-1]),
        "signal_line": _scalar(signal_line.iloc[-1]),
        "rsi": _scalar(rsi.iloc[-1]),
        "high_52w": _scalar(df["High"].max()) if "High" in df.columns else None,
        "low_52w": _scalar(df["Low"].min()) if "Low" in df.columns else None,
    }


print("Market data helpers defined")
print("  - _scalar, fetch_price_history, fetch_fundamentals")
print("  - compute_technical_indicators")

In [ ]:
# ============================================================================
# SHARED: 17-Rule Scoring System (used by Agent 2)
# ============================================================================

def compute_rule_scores(technicals: dict, fundamentals: dict) -> dict:
    """Compute rule-based scores from technical + fundamental data.

    17 individual rules, each adding or subtracting points.
    """
    price = technicals.get("current_price")
    sma50 = technicals.get("sma_50")
    sma200 = technicals.get("sma_200")
    macd = technicals.get("macd")
    signal = technicals.get("signal_line")
    rsi = technicals.get("rsi")
    high52 = technicals.get("high_52w")
    low52 = technicals.get("low_52w")

    pe = fundamentals.get("pe_ratio")
    pb = fundamentals.get("pb_ratio")
    roe = fundamentals.get("roe")
    dy = fundamentals.get("dividend_yield")

    scores = {}
    scores["macd_signal"] = 2 if (macd is not None and signal is not None and macd > signal) else 0
    scores["rsi_bullish"] = 2 if (rsi is not None and 60 < rsi < 70) else 0
    scores["rsi_bearish"] = -2 if (rsi is not None and rsi < 40) else 0
    scores["sma_bull"] = 3 if (sma50 and sma200 and sma50 > sma200) else 0
    scores["sma_bear"] = -3 if (sma50 and sma200 and sma50 < sma200) else 0
    scores["rsi_overbought"] = -2 if (rsi and rsi > 70) else 0
    scores["rsi_neutral_bull"] = 1 if (rsi and 45 < rsi < 60) else 0
    scores["pe_value"] = 2 if (pe and pe < 25) else 0
    scores["pe_high"] = -2 if (pe and pe > 40) else 0
    scores["pb_value"] = 1 if (pb and pb < 3) else 0
    scores["pb_high"] = -1 if (pb and pb > 10) else 0
    scores["roe_strong"] = 2 if (roe and roe > 0.15) else 0
    scores["roe_weak"] = -1 if (roe and roe < 0.05) else 0
    scores["dividend"] = 1 if (dy and dy > 0.01) else 0
    scores["near_high"] = -1 if (price and high52 and price >= high52 * 0.90) else 0
    scores["near_low"] = 1 if (price and low52 and price <= low52 * 1.10) else 0
    scores["total_score"] = sum(scores.values())
    return scores


def rule_recommendation(total_score: int) -> str:
    """Map total score to BUY/HOLD/SELL."""
    if total_score >= 9:
        return "BUY"
    elif total_score >= 0:
        return "HOLD"
    else:
        return "SELL"


def combine_signals(rule_signal: str, llm_signal: str) -> tuple[str, float]:
    """Combine rule-based and LLM signals (60% rules / 40% LLM)."""
    mapping = {"BUY": 1, "HOLD": 0, "SELL": -1}
    score = 0.6 * mapping.get(rule_signal.upper(), 0) + 0.4 * mapping.get(llm_signal.upper(), 0)
    if score >= 0.5:
        return "KEEP / INCREASE", score
    elif score <= -0.5:
        return "REDUCE / SELL", score
    return "MAINTAIN / WATCH", score


def analyze_holding(ticker: str, allocation_pct: float) -> dict:
    """Run complete analysis for a single holding."""
    ticker = ticker.upper().strip()
    fundamentals = fetch_fundamentals(ticker)
    price_df = fetch_price_history(ticker)
    technicals = compute_technical_indicators(price_df)

    if technicals is None:
        return {
            "ticker": ticker,
            "allocation_pct": allocation_pct,
            "data_available": False,
            "company_name": fundamentals.get("company_name", ticker),
            "sector": fundamentals.get("sector", "Unknown"),
        }

    scores = compute_rule_scores(technicals, fundamentals)
    rule_signal = rule_recommendation(scores["total_score"])

    return {
        "ticker": ticker,
        "allocation_pct": allocation_pct,
        "data_available": True,
        "company_name": fundamentals.get("company_name", ticker),
        "sector": fundamentals.get("sector", "Unknown"),
        "current_price": technicals.get("current_price"),
        "sma_50": technicals.get("sma_50"),
        "sma_200": technicals.get("sma_200"),
        "macd": technicals.get("macd"),
        "signal_line": technicals.get("signal_line"),
        "rsi": technicals.get("rsi"),
        "high_52w": technicals.get("high_52w"),
        "low_52w": technicals.get("low_52w"),
        "pe_ratio": fundamentals.get("pe_ratio"),
        "pb_ratio": fundamentals.get("pb_ratio"),
        "roe": fundamentals.get("roe"),
        "dividend_yield": fundamentals.get("dividend_yield"),
        "beta": fundamentals.get("beta"),
        "profit_margins": fundamentals.get("profit_margins"),
        "revenue_growth": fundamentals.get("revenue_growth"),
        "debt_to_equity": fundamentals.get("debt_to_equity"),
        "rule_scores": scores,
        "total_score": scores["total_score"],
        "rule_signal": rule_signal,
    }


print("Scoring & analysis functions defined")
print("  - compute_rule_scores (17 rules)")
print("  - rule_recommendation, combine_signals")
print("  - analyze_holding (full per-holding analysis)")

---
# Agent 1: Portfolio Designer

A conversational profiling agent that asks about your investment goals,
risk tolerance, and preferences, then generates a structured portfolio.
Uses web search for current market context and saves the result to
`portfolio.json` for Agents 2 and 3.

**Architecture:**
```
assistant <------+
   |              |
  tools? -----> ToolNode
   |
  END
```

---

## Step 1: System Prompt & Tools

The profiling agent uses two tools:
- **search_web**: Look up current market data, ETFs, sector performance
- **portfolio_generation**: Save the designed portfolio to JSON

In [ ]:
# ============================================================================
# Agent 1: System Prompt
# ============================================================================
PROFILING_PROMPT = """\
You are an expert investment profiling coach and portfolio designer. \
Your role is to help users build a personalised investment portfolio \
through conversation.

## Your Process
1. **Profile the investor**: Ask about their age, risk tolerance, \
investment goals (growth, income, preservation), time horizon, \
existing investments, and any sector preferences or exclusions.
2. **Gather context**: Use the search_web tool to look up current market \
conditions, ETF options, or sector performance when relevant to the \
user's needs.
3. **Design the portfolio**: Once you have a clear investor profile, \
use the portfolio_generation tool to create and save a structured \
portfolio with specific holdings, allocations, and rationales.

## Profiling Questions to Ask (adapt to context)
- What is your investment time horizon? (e.g., 5, 10, 20+ years)
- What is your risk tolerance? (conservative, moderate, aggressive)
- What are your primary goals? (growth, income, capital preservation)
- Do you prefer ETFs, individual stocks, or a mix?
- Are there sectors you want to emphasise or avoid?
- What is your approximate age and retirement timeline?

## Guidelines
- Be conversational and encouraging -- do not overwhelm with all \
questions at once. Ask 2-3 questions per turn.
- If the user provides enough information upfront (e.g., "I am 45, \
moderate risk, retiring in 15 years"), skip redundant questions and \
move toward portfolio design.
- When searching for market data, summarise key findings concisely.
- When generating a portfolio, include 6-12 holdings with a mix of \
asset types (stocks, ETFs, bonds) appropriate to the investor's profile.
- Allocations must sum to approximately 100%.
- After generating a portfolio, explain your reasoning for each holding \
and offer to adjust it based on feedback.
- Always remind users: this is for educational purposes only, not \
financial advice.
"""

print("Agent 1 system prompt defined")
print(f"  Length: {len(PROFILING_PROMPT)} characters")

## Step 2: Define Tools

Two tools available to the profiling agent:

1. **search_web**: Web search via Serper API for current market data
2. **portfolio_generation**: Creates and saves a structured portfolio to JSON

In [ ]:
# ============================================================================
# Agent 1: Tools
# ============================================================================

@tool
def search_web(query: str) -> str:
    """Search the web for current market data, ETF information,
    sector performance, or investment news."""
    try:
        search = GoogleSerperAPIWrapper()
        return search.run(query)
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def portfolio_generation(
    portfolio_name: str,
    description: str,
    holdings: list[dict],
) -> str:
    """Generate and save a structured investment portfolio.

    Call this when you have gathered enough information about the
    investor's profile and are ready to create a portfolio.

    Args:
        portfolio_name: A descriptive name for the portfolio.
        description: A brief description of the portfolio strategy.
        holdings: List of dicts, each with: ticker, company_name,
            allocation_pct (0-100), investment_type, and rationale.

    Returns:
        Confirmation message with portfolio summary.
    """
    if not holdings:
        return "Holdings list is empty. Please provide at least one holding."

    required_fields = ["ticker", "company_name", "allocation_pct",
                       "investment_type", "rationale"]
    for i, h in enumerate(holdings):
        for field in required_fields:
            if field not in h:
                return f"Holding {i} is missing required field: '{field}'."

    total_alloc = sum(h["allocation_pct"] for h in holdings)
    if total_alloc < 90 or total_alloc > 110:
        return (
            f"Allocations sum to {total_alloc:.1f}%. "
            f"They should sum to approximately 100%. Please adjust."
        )

    portfolio = {
        "name": portfolio_name,
        "description": description,
        "holdings": holdings,
    }

    save_portfolio(portfolio)

    tickers = [h["ticker"] for h in holdings]
    return (
        f"Portfolio '{portfolio_name}' saved with {len(holdings)} holdings "
        f"({', '.join(tickers)}). Total allocation: {total_alloc:.1f}%."
    )


designer_tools = [search_web, portfolio_generation]
print(f"Agent 1 tools ready: {[t.name for t in designer_tools]}")

## Step 3: Build the Agent Graph

Tool-calling agent loop with conditional routing:

```
assistant <------+
   |              |
  tools? -----> ToolNode
   |
  END
```

In [ ]:
# ============================================================================
# Agent 1: Build the Graph
# ============================================================================

designer_llm = load_llm_from_env()
designer_llm_with_tools = designer_llm.bind_tools(designer_tools)


def designer_assistant(state: MessagesState):
    """Call the LLM with the system prompt and conversation history."""
    messages = [SystemMessage(content=PROFILING_PROMPT)] + state["messages"]
    return {"messages": [designer_llm_with_tools.invoke(messages)]}


def designer_should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Route to tools if the LLM made tool calls, otherwise end."""
    return "tools" if state["messages"][-1].tool_calls else "__end__"


designer_graph_def = StateGraph(MessagesState)
designer_graph_def.add_node("assistant", designer_assistant)
designer_graph_def.add_node("tools", ToolNode(designer_tools))
designer_graph_def.add_edge(START, "assistant")
designer_graph_def.add_conditional_edges(
    "assistant", designer_should_continue, ["tools", "__end__"]
)
designer_graph_def.add_edge("tools", "assistant")

designer_memory = MemorySaver()
designer_graph = designer_graph_def.compile(checkpointer=designer_memory)

print("Agent 1: Portfolio Designer compiled")

In [ ]:
try:
    display(Image(designer_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph visualisation requires extra dependencies)")

## Step 4: Chat Helper & Demo

Use `designer_chat()` to talk to the profiling agent.
Use the same `thread_id` to continue a conversation.

In [ ]:
def designer_chat(user_input: str, thread_id: str = "default") -> str:
    """Chat with the portfolio design agent."""
    config = {"configurable": {"thread_id": thread_id}}

    result = None
    for event in designer_graph.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
        stream_mode="values",
    ):
        result = event

    return result["messages"][-1].content


# Demo: Send a portfolio construction query
print("=" * 70)
print("AGENT 1: PORTFOLIO DESIGNER")
print("=" * 70)

print("\nUser: I am 45, moderate risk, retiring in 15 years -- what allocation should I use?")
response = designer_chat(
    "I am 45, moderate risk, retiring in 15 years -- what allocation should I use?",
    thread_id="demo",
)
print(f"\nBot: {response}")

In [ ]:
# Verify saved portfolio
if os.path.exists(PORTFOLIO_FILE):
    with open(PORTFOLIO_FILE) as f:
        saved = json.load(f)
    save_portfolio(saved)  # Update SHARED_PORTFOLIO for Agents 2 & 3
    show_portfolio()
else:
    print(f"No portfolio file found at {PORTFOLIO_FILE}")
    print("Continue chatting with Agent 1 to generate a portfolio.")

In [ ]:
# Interactive profiling session (optional)
print("=" * 70)
print("AGENT 1: INTERACTIVE PROFILING")
print("=" * 70)
print("Tell me about your investment goals and I'll design a portfolio.")
print("Type 'quit' or 'exit' to stop.")
print("=" * 70)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nChat ended.")
        break

    response = designer_chat(user_input, thread_id="interactive")
    print(f"\nAI: {response}")

---
# Agent 2: Portfolio Analytics

The full analytics agent from Notebook 12. Reads the portfolio created by
Agent 1 and runs the complete 17-rule scoring system plus LLM structured
signals with 60/40 combined recommendations.

**Pipeline:** `load_portfolio -> build_benchmark_context -> analyze_holdings -> compute_risk -> generate_analysis -> END`

---

In [ ]:
# ============================================================================
# Agent 2: State, Structured Output, and Prompt
# ============================================================================

class AnalyticsState(BaseModel):
    """State for the analytics agent graph."""
    model_config = {"arbitrary_types_allowed": True}
    messages: Annotated[Sequence[BaseMessage], add_messages] = []
    portfolio: dict = {}
    benchmark_context: str = ""
    sector_rows: list[dict] = []
    holdings_analysis: list[dict] = []
    risk_metrics: dict = {}


class HoldingSignal(BaseModel):
    """LLM recommendation for a single holding."""
    ticker: str = Field(description="Stock ticker symbol")
    signal: str = Field(description="BUY, HOLD, or SELL")


class HoldingSignals(BaseModel):
    """Structured LLM output: per-holding BUY/HOLD/SELL signals."""
    signals: list[HoldingSignal]


ANALYTICS_PROMPT = """You are an expert investment analyst evaluating a portfolio. \
You have been provided with:

1. **Sector weight comparison** against the S&P 500 benchmark
2. **Per-holding analysis** with live technical indicators (RSI, MACD, SMA), \
fundamentals (P/E, P/B, ROE, beta), and rule-based BUY/HOLD/SELL scores
3. **Portfolio-level risk metrics** (beta, concentration, sector breakdown)

Produce a comprehensive analysis covering ALL of the following:

## Sector Exposure
- Which sectors are overweight/underweight vs S&P 500
- Concentration risks in specific sectors

## Individual Holdings Review
- Flag holdings with SELL signals or very low scores
- Highlight strong BUY-signal holdings
- Note any overbought (RSI >70) or oversold (RSI <30) positions

## Risk Assessment
- Portfolio beta and volatility exposure
- Concentration risk (single-stock and sector)
- Valuation risk (average P/E vs market)

## ROI & Performance Indicators
- Holdings with strong momentum (MACD bullish, RSI 50-70)
- Dividend income potential
- Growth vs value tilt

## Rebalancing Recommendations
- Specific positions to reduce or increase
- Sector adjustments needed to improve diversification
- Priority actions ranked by urgency

Format in markdown with clear sections. Be specific — use ticker symbols, \
percentages, and scores from the data provided. Keep each section concise."""

print("Agent 2 state and structured output models defined")

In [ ]:
# ============================================================================
# Agent 2: LLM Structured Signals
# ============================================================================

def get_llm_signals(holdings_summary: list[dict], llm) -> list[dict] | None:
    """Ask the LLM for per-holding BUY/HOLD/SELL signals.

    Uses with_structured_output for typed Pydantic responses.
    Combines LLM signal with rule signal using 60/40 weighting.
    """
    tickers_with_data = [
        h for h in holdings_summary
        if h.get("data_available", True) and h.get("rule_signal")
    ]
    if not tickers_with_data:
        return None

    prompt = (
        "For each stock below, give a BUY, HOLD, or SELL recommendation "
        "based on the technical and fundamental data provided.\n\n"
        + json.dumps(tickers_with_data, indent=2, default=str)
    )

    try:
        structured_llm = llm.with_structured_output(HoldingSignals)
        result = structured_llm.invoke([HumanMessage(content=prompt)])

        signal_map = {s.ticker.upper(): s.signal.upper() for s in result.signals}

        for h in holdings_summary:
            ticker = h["ticker"].upper()
            if ticker in signal_map and h.get("rule_signal"):
                h["llm_signal"] = signal_map[ticker]
                combined_rec, combined_score = combine_signals(
                    h["rule_signal"], signal_map[ticker]
                )
                h["combined_signal"] = combined_rec
                h["combined_score"] = round(combined_score, 3)

        return holdings_summary
    except Exception as e:
        print(f"    Warning: Failed to get LLM signals: {e}")
        return None


print("LLM structured signals function defined")

In [ ]:
# ============================================================================
# Agent 2: Pipeline Nodes
# ============================================================================

def analytics_load(state: AnalyticsState) -> dict:
    """Node 1: Load portfolio from shared state."""
    print("  [Node 1] Loading portfolio...")
    portfolio = SHARED_PORTFOLIO
    if not portfolio:
        portfolio = load_portfolio_from_file()
    if not portfolio or not portfolio.get("holdings"):
        print("    No portfolio found. Run Agent 1 first.")
        return {"portfolio": {}}
    print(f"    '{portfolio['name']}' with {len(portfolio['holdings'])} holdings")
    return {"portfolio": portfolio}


def analytics_benchmark(state: AnalyticsState) -> dict:
    """Node 2: Compare portfolio sectors against S&P 500."""
    print("  [Node 2] Building benchmark context...")
    holdings = state.portfolio.get("holdings", [])
    if not holdings:
        return {"benchmark_context": "", "sector_rows": []}

    sector_weights = {}
    lines = ["PORTFOLIO WEIGHT ANALYSIS DATA\n", "Portfolio Holdings:"]
    for h in holdings:
        fundamentals = fetch_fundamentals(h["ticker"])
        sector = fundamentals.get("sector", "Unknown")
        company = fundamentals.get("company_name", h["ticker"])
        sector_weights[sector] = sector_weights.get(sector, 0) + h["allocation_pct"]
        lines.append(f"  {h['ticker']} ({company}): {h['allocation_pct']:.1f}% - Sector: {sector}")

    lines.append("\nSector Allocation Comparison (Portfolio vs S&P 500 Benchmark):")
    all_sectors = sorted(set(list(SP500_SECTOR_WEIGHTS.keys()) + list(sector_weights.keys())))
    rows = []
    for sector in all_sectors:
        port_w = sector_weights.get(sector, 0)
        sp_w = SP500_SECTOR_WEIGHTS.get(sector, 0)
        diff = port_w - sp_w
        status = "OVERWEIGHT" if diff > 5 else "UNDERWEIGHT" if diff < -5 else "NEUTRAL"
        lines.append(f"  {sector}: Portfolio {port_w:.1f}% vs S&P 500 {sp_w:.1f}% (Difference: {diff:+.1f}%)")
        rows.append({"sector": sector, "portfolio_weight": round(port_w, 1),
                      "benchmark_weight": round(sp_w, 1), "difference": round(diff, 1), "status": status})

    context = "\n".join(lines)
    print(f"    Compared {len(rows)} sectors")
    return {"benchmark_context": context, "sector_rows": rows}


def analytics_analyze(state: AnalyticsState) -> dict:
    """Node 3: Fetch live data and compute per-holding analysis."""
    print("  [Node 3] Analysing holdings (live data + 17-rule scoring)...")
    holdings = state.portfolio.get("holdings", [])
    if not holdings:
        return {"holdings_analysis": []}

    analyses = []
    for h in holdings:
        print(f"    -> {h['ticker']} ({h['allocation_pct']}%)...", end=" ")
        result = analyze_holding(h["ticker"], h["allocation_pct"])
        if result["data_available"]:
            print(f"Score: {result['total_score']} [{result['rule_signal']}]")
        else:
            print("(no data)")
        analyses.append(result)

    print(f"    {len(analyses)} holdings analysed")
    return {"holdings_analysis": analyses}


def analytics_risk(state: AnalyticsState) -> dict:
    """Node 4: Compute portfolio-level risk metrics."""
    print("  [Node 4] Computing risk metrics...")
    if not state.holdings_analysis:
        return {"risk_metrics": {"overall_risk_level": "UNKNOWN"}}

    available = [h for h in state.holdings_analysis if h.get("data_available")]
    if not available:
        return {"risk_metrics": {"overall_risk_level": "UNKNOWN"}}

    betas = [(h["beta"], h["allocation_pct"]) for h in available if h.get("beta") is not None]
    avg_beta = sum(b * w for b, w in betas) / sum(w for _, w in betas) if betas else None
    pes = [(h["pe_ratio"], h["allocation_pct"]) for h in available if h.get("pe_ratio") is not None]
    avg_pe = sum(p * w for p, w in pes) / sum(w for _, w in pes) if pes else None

    weights = [h["allocation_pct"] for h in available]
    hhi = sum((w / 100) ** 2 for w in weights)
    max_weight = max(weights)
    sector_alloc = {}
    for h in available:
        s = h.get("sector", "Unknown")
        sector_alloc[s] = sector_alloc.get(s, 0) + h["allocation_pct"]

    high_risk = [h for h in available if h.get("rule_signal") == "SELL"]
    risk_factors = 0
    if avg_beta is not None and avg_beta > 1.3: risk_factors += 1
    if max_weight > 25: risk_factors += 1
    if len(high_risk) > len(available) * 0.3: risk_factors += 1
    if hhi > 0.15: risk_factors += 1
    risk_level = "HIGH" if risk_factors >= 3 else "MODERATE" if risk_factors >= 1 else "LOW"

    metrics = {
        "overall_risk_level": risk_level,
        "weighted_avg_beta": round(avg_beta, 3) if avg_beta else None,
        "weighted_avg_pe": round(avg_pe, 2) if avg_pe else None,
        "concentration_hhi": round(hhi, 4),
        "max_single_holding_pct": round(max_weight, 1),
        "sector_breakdown": {k: round(v, 1) for k, v in sorted(sector_alloc.items(), key=lambda x: -x[1])},
        "high_risk_holdings": [h["ticker"] for h in high_risk],
    }
    print(f"    Beta: {metrics['weighted_avg_beta']}, P/E: {metrics['weighted_avg_pe']}, Risk: {risk_level}")
    return {"risk_metrics": metrics}


def analytics_generate(state: AnalyticsState) -> dict:
    """Node 5: LLM analysis + structured per-holding signals."""
    print("  [Node 5] Generating LLM analysis...")
    if not state.portfolio:
        return {"messages": [SystemMessage(content="No portfolio data available.")]}

    holdings_summary = []
    for h in state.holdings_analysis:
        entry = {"ticker": h["ticker"], "allocation_pct": h["allocation_pct"], "sector": h.get("sector", "Unknown")}
        if h.get("data_available"):
            entry.update({
                "current_price": h.get("current_price"),
                "rsi": round(h["rsi"], 1) if h.get("rsi") else None,
                "macd": round(h["macd"], 4) if h.get("macd") else None,
                "signal_line": round(h["signal_line"], 4) if h.get("signal_line") else None,
                "sma_50": round(h["sma_50"], 2) if h.get("sma_50") else None,
                "sma_200": round(h["sma_200"], 2) if h.get("sma_200") else None,
                "pe_ratio": h.get("pe_ratio"), "pb_ratio": h.get("pb_ratio"),
                "roe": h.get("roe"), "beta": h.get("beta"),
                "dividend_yield": h.get("dividend_yield"),
                "total_score": h.get("total_score"), "rule_signal": h.get("rule_signal"),
            })
        else:
            entry["data_available"] = False
        holdings_summary.append(entry)

    parts = []
    if state.benchmark_context:
        parts.append(state.benchmark_context)
    parts.append("\nPER-HOLDING ANALYSIS:")
    parts.append(json.dumps(holdings_summary, indent=2, default=str))
    parts.append("\nPORTFOLIO RISK METRICS:")
    parts.append(json.dumps(state.risk_metrics, indent=2, default=str))

    llm = load_llm_from_env()

    # Comprehensive analysis
    messages = [SystemMessage(content=ANALYTICS_PROMPT), HumanMessage(content="\n".join(parts))]
    response = llm.invoke(messages)
    print("    Comprehensive analysis generated")

    # Structured LLM signals + combined 60/40
    print("    -> Getting structured LLM signals...")
    updated = get_llm_signals(holdings_summary, llm)
    if updated:
        print("    Combined signals computed (60% rules / 40% LLM)")
        return {"messages": [response], "holdings_analysis": updated}
    else:
        print("    Using rule-based signals only")
        return {"messages": [response]}


print("Agent 2 nodes defined")

In [ ]:
# ============================================================================
# Agent 2: Build the Graph
# ============================================================================

analytics_graph_def = StateGraph(AnalyticsState)
analytics_graph_def.add_node("load_portfolio", analytics_load)
analytics_graph_def.add_node("build_benchmark_context", analytics_benchmark)
analytics_graph_def.add_node("analyze_holdings", analytics_analyze)
analytics_graph_def.add_node("compute_risk", analytics_risk)
analytics_graph_def.add_node("generate_analysis", analytics_generate)

analytics_graph_def.set_entry_point("load_portfolio")
analytics_graph_def.add_edge("load_portfolio", "build_benchmark_context")
analytics_graph_def.add_edge("build_benchmark_context", "analyze_holdings")
analytics_graph_def.add_edge("analyze_holdings", "compute_risk")
analytics_graph_def.add_edge("compute_risk", "generate_analysis")
analytics_graph_def.add_edge("generate_analysis", END)

analytics_graph = analytics_graph_def.compile()

print("Agent 2: Analytics pipeline compiled")

In [ ]:
try:
    display(Image(analytics_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph visualisation requires extra dependencies)")

## Run the Analytics Agent

This reads the portfolio from Agent 1 and runs the full analytics pipeline:
17-rule scoring, risk metrics, LLM analysis, and structured LLM signals
with 60/40 combined recommendations.

In [ ]:
print("=" * 70)
print("AGENT 2: RUNNING PORTFOLIO ANALYTICS AGENT")
print("=" * 70)

analytics_result = analytics_graph.invoke(AnalyticsState())

print("\n" + "=" * 70)
print("Pipeline complete!")
print("=" * 70)

In [ ]:
# View the LLM analysis
display(Markdown(analytics_result["messages"][-1].content))

In [ ]:
# View combined signals (Rule + LLM)
print("=" * 90)
print(f"{'Ticker':<7} {'Alloc':>5} {'Price':>10} {'RSI':>6} {'Score':>6} "
      f"{'Rule':>6} {'LLM':>6} {'Combined':<20}")
print("=" * 90)
for h in analytics_result["holdings_analysis"]:
    ticker = h.get("ticker", "?")
    alloc = f"{h.get('allocation_pct', 0):.1f}%"
    price = f"${h['current_price']:.2f}" if h.get('current_price') else 'N/A'
    rsi = f"{h['rsi']:.1f}" if h.get('rsi') else 'N/A'
    score = h.get('total_score', 'N/A')
    rule = h.get('rule_signal', 'N/A')
    llm = h.get('llm_signal', '-')
    combined = h.get('combined_signal', '-')
    print(f"  {ticker:<6} {alloc:>5} {price:>10} {rsi:>6} {score:>6} "
          f"{rule:>6} {llm:>6} {combined:<20}")

In [ ]:
# View risk metrics
metrics = analytics_result["risk_metrics"]
print("=" * 70)
print("PORTFOLIO RISK METRICS")
print("=" * 70)
print(f"  Overall Risk Level:     {metrics.get('overall_risk_level', 'N/A')}")
print(f"  Weighted Avg Beta:      {metrics.get('weighted_avg_beta', 'N/A')}")
print(f"  Weighted Avg P/E:       {metrics.get('weighted_avg_pe', 'N/A')}")
print(f"  Concentration HHI:      {metrics.get('concentration_hhi', 'N/A')}")
print(f"  Max Single Holding:     {metrics.get('max_single_holding_pct', 'N/A')}%")
print(f"  High-Risk Holdings:     {metrics.get('high_risk_holdings', [])}")
print(f"\n  Sector Breakdown:")
for sector, pct in metrics.get('sector_breakdown', {}).items():
    print(f"    {sector:<25} {pct:>5.1f}%")

---
# Agent 3: Investment Education Chat

The education agent from Notebook 13. Uses the portfolio from Agent 1
to personalise responses. Supports multi-turn conversation — ask
follow-up questions about your portfolio and investment concepts.

**Pipeline:** `enrich_context -> call_model -> END`

---

In [ ]:
# ============================================================================
# Agent 3: State & Prompt
# ============================================================================

class EducationState(BaseModel):
    """State for the education agent graph."""
    model_config = {"arbitrary_types_allowed": True}
    messages: Annotated[Sequence[BaseMessage], add_messages] = []
    system_prompt: str = ""
    portfolio_context: str = ""


EDUCATION_PROMPT = """You are an expert investment education assistant. \
Your role is to help users understand investment concepts, portfolio \
management, and financial markets.

Guidelines:
- Explain concepts clearly using plain language
- Use real-world examples when possible
- If the user has a portfolio, reference their specific holdings to \
make explanations more relevant and personalised
- Cover topics like: asset allocation, diversification, risk management, \
technical analysis (RSI, MACD, SMA), fundamental analysis (P/E, P/B, ROE), \
sector rotation, rebalancing, dividend investing, growth vs value
- Always remind users that this is for educational purposes only and \
not financial advice
- When discussing the user's portfolio, explain the \"why\" behind \
observations (e.g. why being overweight in tech increases volatility)
- Be conversational and encourage follow-up questions"""

print("Agent 3 state and prompt defined")

In [ ]:
# ============================================================================
# Agent 3: Sector Comparison Helper
# ============================================================================

def compute_sector_comparison(holdings: list[dict]) -> str:
    """Compare portfolio sector weights against S&P 500."""
    if not holdings:
        return ""

    sector_weights = {}
    for h in holdings:
        try:
            info = yf.Ticker(h["ticker"]).info
            sector = info.get("sector", "Unknown")
        except Exception:
            sector = "Unknown"
        sector_weights[sector] = sector_weights.get(sector, 0) + h.get("allocation_pct", 0)

    lines = ["Sector Allocation Comparison (Portfolio vs S&P 500):"]
    all_sectors = sorted(set(list(SP500_SECTOR_WEIGHTS.keys()) + list(sector_weights.keys())))

    for sector in all_sectors:
        port_w = sector_weights.get(sector, 0)
        sp_w = SP500_SECTOR_WEIGHTS.get(sector, 0)
        diff = port_w - sp_w
        if port_w > 0 or abs(diff) > 3:
            status = "OVERWEIGHT" if diff > 5 else "UNDERWEIGHT" if diff < -5 else "NEUTRAL"
            lines.append(
                f"  {sector}: Portfolio {port_w:.1f}% vs S&P 500 {sp_w:.1f}% "
                f"({diff:+.1f}%) [{status}]"
            )

    lines.append("\nOverweight = more than S&P 500 benchmark; underweight = less.")
    return "\n".join(lines)


print("Sector comparison helper defined")

In [ ]:
# ============================================================================
# Agent 3: Pipeline Nodes
# ============================================================================

def education_enrich(state: EducationState) -> dict:
    """Node 1: Build enriched system prompt with portfolio context."""
    base_prompt = EDUCATION_PROMPT
    portfolio_context = ""

    portfolio = SHARED_PORTFOLIO
    if not portfolio:
        portfolio = load_portfolio_from_file()

    if portfolio and portfolio.get("holdings"):
        holdings = portfolio["holdings"]
        tickers = [h["ticker"] for h in holdings]

        portfolio_context = (
            f"USER'S CURRENT PORTFOLIO:\n"
            f"Securities: {', '.join(tickers)}.\n"
        )

        allocations = [
            f"{h['ticker']} ({h.get('allocation_pct', 0)}%)"
            for h in holdings if h.get("allocation_pct")
        ]
        if allocations:
            portfolio_context += f"Allocations: {', '.join(allocations)}.\n"

        holdings_for_comparison = [
            {"ticker": h["ticker"], "allocation_pct": h.get("allocation_pct", 0)}
            for h in holdings
        ]
        sector_text = compute_sector_comparison(holdings_for_comparison)
        if sector_text:
            portfolio_context += f"\n{sector_text}\n"

    full_prompt = base_prompt + (f"\n\n{portfolio_context}" if portfolio_context else "")
    return {"system_prompt": full_prompt, "portfolio_context": portfolio_context}


def education_call_model(state: EducationState) -> dict:
    """Node 2: Call the LLM with system prompt + conversation history."""
    llm = load_llm_from_env()
    messages_for_llm = [SystemMessage(content=state.system_prompt)] + list(state.messages)
    response = llm.invoke(messages_for_llm)
    return {"messages": [response]}


print("Agent 3 nodes defined")

In [ ]:
# ============================================================================
# Agent 3: Build the Graph
# ============================================================================

education_graph_def = StateGraph(EducationState)
education_graph_def.add_node("enrich_context", education_enrich)
education_graph_def.add_node("call_model", education_call_model)

education_graph_def.set_entry_point("enrich_context")
education_graph_def.add_edge("enrich_context", "call_model")
education_graph_def.add_edge("call_model", END)

education_graph = education_graph_def.compile()

print("Agent 3: Education pipeline compiled")
print("  enrich_context -> call_model -> END")

In [ ]:
try:
    display(Image(education_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(Graph visualisation requires extra dependencies)")

In [ ]:
# ============================================================================
# Agent 3: Multi-Turn Chat Wrapper
# ============================================================================

class EducationChat:
    """Multi-turn chat wrapper around the education agent graph."""

    def __init__(self, graph):
        self.graph = graph
        self.conversation_history: list[BaseMessage] = []

    def ask(self, question: str) -> str:
        """Send a question and get a response."""
        self.conversation_history.append(HumanMessage(content=question))
        result = self.graph.invoke(
            EducationState(messages=self.conversation_history)
        )
        ai_message = result["messages"][-1]
        self.conversation_history.append(ai_message)
        return ai_message.content

    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("Conversation history cleared")

    def show_history(self):
        """Display conversation history."""
        print("=" * 70)
        print("CONVERSATION HISTORY")
        print("=" * 70)
        for i, msg in enumerate(self.conversation_history):
            role = "USER" if isinstance(msg, HumanMessage) else "AI"
            print(f"\n[{i+1}] {role}:")
            print(msg.content[:200] + ("..." if len(msg.content) > 200 else ""))
        print("=" * 70)


# Create the chat instance
chat = EducationChat(education_graph)

print("Education chat ready")
print("  Usage: response = chat.ask('your question here')")
print("  Reset: chat.reset()")
print("  History: chat.show_history()")

## Demo: Single Question

Try a single question. If you created a portfolio in Agent 1,
the answer will reference your specific holdings.

In [ ]:
response = chat.ask("What is diversification and why does it matter?")
display(Markdown(response))

## Multi-Turn Conversation

Ask follow-up questions. The agent remembers your earlier exchanges
and references your portfolio holdings.

In [ ]:
response = chat.ask("Based on my portfolio, would you say I lean more towards growth or value?")
display(Markdown(response))

In [ ]:
response = chat.ask("What is RSI and how would I use it to evaluate my holdings?")
display(Markdown(response))

In [ ]:
response = chat.ask(
    "You mentioned growth vs value earlier. "
    "How does sector concentration relate to that? "
    "Am I too concentrated in any one sector?"
)
display(Markdown(response))

In [ ]:
chat.show_history()

## Interactive Chat Loop

Free-form interactive chat session. Type `quit` or `exit` to stop.

In [ ]:
# Reset for a fresh interactive session
chat.reset()

print("=" * 70)
print("INVESTMENT EDUCATION CHAT")
print("=" * 70)
if SHARED_PORTFOLIO:
    tickers = [h['ticker'] for h in SHARED_PORTFOLIO['holdings']]
    print(f"Portfolio loaded: {', '.join(tickers)}")
else:
    print("No portfolio loaded (general education mode)")
print("Type 'quit' or 'exit' to stop.")
print("=" * 70)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nChat ended. Use chat.show_history() to review.")
        break

    response = chat.ask(user_input)
    print(f"\nAI: {response}")

---
# Re-Run with a Different Portfolio

To redesign your portfolio and re-analyse:
1. Chat with **Agent 1** again (use the interactive cell or `designer_chat()`)
2. Once a new portfolio is saved, re-run **Agent 2** to get updated analytics
3. Reset the education chat (`chat.reset()`) and ask new questions in **Agent 3**

---

In [ ]:
# ============================================================================
# QUICK SWAP: Redesign portfolio with a single query
# ============================================================================
# Use designer_chat() with a detailed profile to get a portfolio in one turn:

# response = designer_chat(
#     "Build me a conservative income portfolio. I am 60, retiring in 5 years, "
#     "prefer ETFs and dividend stocks, no crypto or speculative tech.",
#     thread_id="quick-swap",
# )
# print(response)
#
# # Load the new portfolio into shared state
# with open(PORTFOLIO_FILE) as f:
#     save_portfolio(json.load(f))
# show_portfolio()
#
# # Re-run Agent 2
# analytics_result = analytics_graph.invoke(AnalyticsState())
# display(Markdown(analytics_result["messages"][-1].content))
#
# # Reset Agent 3 chat
# chat.reset()

## Example Questions for Agent 3

**General concepts:**
- "What is dollar-cost averaging?"
- "Explain the difference between ETFs and mutual funds"
- "What is a golden cross in technical analysis?"

**Portfolio-specific (if portfolio loaded):**
- "Is my portfolio well diversified?"
- "Which of my holdings has the most risk?"
- "Should I add bonds to balance my portfolio?"
- "What would happen to my portfolio in a recession?"

**Follow-up questions:**
- "Can you explain that in simpler terms?"
- "How does that apply to my specific holdings?"
- "What would you recommend I learn next?"

---
## Summary

- **Investment Agent Hub** — three agents in one notebook
- **Agent 1 (Designer)**: Conversational profiling chatbot with web search + portfolio generation tools
- **Agent 2 (Analytics)**: Full 17-rule scoring + LLM structured signals (60/40 combined)
- **Agent 3 (Education)**: Multi-turn chat with portfolio-aware context enrichment
- **Shared portfolio**: In-memory variable + JSON file for persistence

### Agent Architecture
```
Agent 1 (Designer):    assistant <-> ToolNode (agent loop with conditional edges)
Agent 2 (Analytics):   load -> benchmark -> analyze -> risk -> generate -> END
Agent 3 (Education):   enrich_context -> call_model -> END
```

### Data Flow
```
Agent 1 (Designer)  ──save_portfolio()──> SHARED_PORTFOLIO + portfolio.json
                                                │
                         ┌──────────────────────┤
                         │                      │
                         ▼                      ▼
              Agent 2 (Analytics)     Agent 3 (Education)
              Full scoring report     Interactive chat
```

### Key Features
| Feature | Agent 1 | Agent 2 | Agent 3 |
|---------|---------|---------|----------|
| Graph shape | Agent loop | Linear (5 nodes) | Linear (2 nodes) |
| LLM calls | Multiple per conversation | 2 (analysis + signals) | 1 per turn |
| Tools | search_web, portfolio_generation | None | None |
| Conditional edges | Yes (tool loop) | No | No |
| Output | portfolio.json | One-shot + combined signals | Multi-turn chat |
| Interaction | Multi-turn profiling | Run once | Conversation |